In [1]:
from dataclasses import dataclass
import base64
import os
import httpx
import threading

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import copy
from tqdm import tqdm


def encode_audio_to_base64(audio_path: str) -> str:
    """Encode a local audio file to base64 data URL."""
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    # Detect MIME type from extension
    audio_path_lower = audio_path.lower()
    if audio_path_lower.endswith(".wav"):
        mime_type = "audio/wav"
    elif audio_path_lower.endswith((".mp3", ".mpeg")):
        mime_type = "audio/mpeg"
    elif audio_path_lower.endswith(".flac"):
        mime_type = "audio/flac"
    elif audio_path_lower.endswith(".ogg"):
        mime_type = "audio/ogg"
    else:
        mime_type = "audio/wav"  # Default

    with open(audio_path, "rb") as f:
        audio_bytes = f.read()
    audio_b64 = base64.b64encode(audio_bytes).decode("utf-8")
    return f"data:{mime_type};base64,{audio_b64}"

In [2]:
@dataclass
class Args:
    api_base:str = "http://localhost:8091"
    output: str = "tts_output.wav"
    model: str = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
    text: str = "Я люблю свою работу. Каждый день я узнаю что-то новое - и это очень, очень круто!"
    voice: str = None
    ref_audio: str = "/lustre/users/rkoshkin/s2st/assets/ru_ref.sample.wav"
    ref_text: str = "Привет! С вами Программный Комитет - шоу подкаст-студии Термин-Вокс и IT-конфереции Стачка. Меня зовут Сергей Пихин.В этом подкасте мы обсуждаем главные тренды в IT-индустрии и в смежных областях. Помогают нам в этом топовые эксперты, которые делятся своими знаниями и экспертизой."
    response_format: str = "wav"
    task_type: str = "Base"
    instructions: str = None
    language: str = "Russian"
    max_new_tokens: str = None
    x_vector_only: str = None
    api_key: str = "EMPTY"


def run_tts_generation(args) -> None:
    """Run TTS generation via OpenAI-compatible /v1/audio/speech API."""

    # Build request payload
    payload = {
        "model": args.model,
        "input": args.text,
        "voice": args.voice,
        "response_format": args.response_format,
    }

    # Add optional parameters
    if args.instructions:
        payload["instructions"] = args.instructions
    if args.task_type:
        payload["task_type"] = args.task_type
    if args.language:
        payload["language"] = args.language
    if args.max_new_tokens:
        payload["max_new_tokens"] = args.max_new_tokens

    # Voice clone parameters (Base task)
    if args.ref_audio:
        if args.ref_audio.startswith(("http://", "https://")):
            payload["ref_audio"] = args.ref_audio
        else:
            payload["ref_audio"] = encode_audio_to_base64(args.ref_audio)
    if args.ref_text:
        payload["ref_text"] = args.ref_text
    if args.x_vector_only:
        payload["x_vector_only_mode"] = True

    # Make the API call
    api_url = f"{args.api_base}/v1/audio/speech"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {args.api_key}",
    }

    with httpx.Client(timeout=300.0) as client:
        response = client.post(api_url, json=payload, headers=headers)

    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return

    # Check for JSON error response (only if content is valid UTF-8 text)
    try:
        text = response.content.decode("utf-8")
        if text.startswith('{"error"'):
            print(f"Error: {text}")
            return
    except UnicodeDecodeError:
        pass  # Binary audio data, not an error

    with open(args.output, "wb") as f:
        f.write(response.content)

In [3]:
from datasets import Dataset, DatasetDict

ds = DatasetDict.load_from_disk("/lustre/users/rkoshkin/s2st/data/s2s/podcast_crawl-enru-dd-full-s2s+sid/")['train']

Loading dataset from disk:   0%|          | 0/97 [00:00<?, ?it/s]

In [ ]:
%%time

def _send_request(i):
    args = Args(output=f"tts_output_{i}.wav", text=sample['tgt_sent'][i])
    run_tts_generation(args)


sample = ds[0]
with ThreadPoolExecutor(max_workers=100) as executor:
    results = list(tqdm(
        executor.map(_send_request, range(180)),
        total=180,
        desc="Generating TTS"
    ))

Generating TTS:   2%|███▍                                                                                                                                                       | 4/180 [00:45<32:28, 11.07s/it]

In [5]:
!ls | grep tts_ | xargs rm

rm: missing operand
Try 'rm --help' for more information.


# Batched inference

In [5]:
import os
from typing import NamedTuple
import soundfile as sf
from typing import List

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import SamplingParams
from vllm.utils.argparse_utils import FlexibleArgumentParser
from vllm_omni import Omni

class QueryResult(NamedTuple):
    """Container for a prepared Omni request."""

    inputs: dict
    model_name: str


omni = Omni(
    model="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    stage_configs_path=None,
    log_stats=True,
    stage_ibnit_timeout=300,
)

/lustre/users/rkoshkin/vllm-omni/.venv/lib/python3.10/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


INFO 02-21 00:52:33 [weight_utils.py:50] Using model weights format ['*']
INFO 02-21 00:52:33 [omni.py:138] Initializing stages for model: Qwen/Qwen3-TTS-12Hz-1.7B-Base


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-21 00:52:34 [initialization.py:234] Loaded OmniTransferConfig with 0 connector configurations
INFO 02-21 00:52:34 [omni_stage.py:249] [OmniStage] stage_config: {'stage_id': 0, 'stage_type': 'llm', 'runtime': {'devices': '0', 'max_batch_size': 100}, 'engine_args': {'model_stage': 'qwen3_tts', 'model_arch': 'Qwen3TTSForConditionalGeneration', 'worker_type': 'generation', 'scheduler_cls': 'vllm_omni.core.sched.omni_generation_scheduler.OmniGenerationScheduler', 'enforce_eager': True, 'trust_remote_code': True, 'async_scheduling': False, 'enable_prefix_caching': False, 'engine_output_type': 'audio', 'gpu_memory_utilization': 0.8, 'distributed_executor_backend': 'mp', 'max_num_batched_tokens': 1000000, 'max_num_seqs': 100, 'async_chunk': False}, 'final_output': True, 'final_output_type': 'audio'}
INFO 02-21 00:52:34 [omni.py:357] [Orchestrator] Waiting for 1 stages to initialize (timeout: 300s)


/lustre/users/rkoshkin/vllm-omni/.venv/lib/python3.10/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


...

(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:54 [configuration_qwen3_tts.py:489] talker_config is None. Initializing talker model with default values
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:54 [configuration_qwen3_tts.py:492] speaker_encoder_config is None. Initializing talker model with default values
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:54 [configuration_qwen3_tts.py:441] code_predictor_config is None. Initializing code_predictor model with default values
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:54 [configuration_qwen3_tts.py:441] code_predictor_config is None. Initializing code_predictor model with default values
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:55 [weight_utils.py:50] Using model weights format ['speech_tokenizer/*']
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:56 [configuration_qwen3_tts_tokenizer_v2.py:156] encoder_config is None. Initializing encoder with default values
(Worker pid=1593152) [Stage-0] INFO 02-21 00:53:56 [configurati

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00, 63.75it/s]
(Worker pid=1593152) 


(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:01 [gpu_model_runner.py:4221] Model loading took 3.92 GiB memory and 6.429910 seconds
(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:01 [kernel_warmup.py:44] Skipping FlashInfer autotune because it is disabled.
(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:01 [qwen3_tts.py:133] Profile run detected (empty text). Capping max_new_tokens to 2.
(Worker pid=1593152) [Stage-0] WARNING 02-21 00:54:01 [qwen3_tts.py:804] ref_audio is not provided. Using a 1-second silent clip to satisfy padding requirements. Please check if it is profile run or you missed to provide ref_audio.
(Worker pid=1593152) [Stage-0] WARNING 02-21 00:54:01 [qwen3_tts.py:682] ref_text is required when x_vector_only_mode=False (ICL mode). Bad index=0. Please check if it is profile run or you missed to provide ref_text.
(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:02 [configuration_qwen3_tts.py:441] code_predictor_config is None. Initializing code_predictor model with de

(Worker pid=1593152) Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


(Worker pid=1593152) [Stage-0] WARNING 02-21 00:54:02 [gpu_generation_model_runner.py:449] Dummy sampler run is not implemented for generation model
(EngineCore_DP0 pid=1592350) [Stage-0] INFO 02-21 00:54:02 [core.py:278] init engine (profile, create kv cache, warmup model) took 1.28 seconds
(EngineCore_DP0 pid=1592350) [Stage-0] WARNING 02-21 00:54:04 [scheduler.py:166] Using custom scheduler class vllm_omni.core.sched.omni_generation_scheduler.OmniGenerationScheduler. This scheduler interface is not public and compatibility may not be maintained.
(EngineCore_DP0 pid=1592350) [Stage-0] WARNING 02-21 00:54:04 [core.py:130] Disabling chunked prefill for model without KVCache
(EngineCore_DP0 pid=1592350) [Stage-0] INFO 02-21 00:54:04 [vllm.py:689] Asynchronous scheduling is disabled.
(EngineCore_DP0 pid=1592350) [Stage-0] WARNING 02-21 00:54:04 [vllm.py:734] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be

In [ ]:
%%time


def get_base_query(
    ref_audios: List[str],
    ref_texts: List[str],
    target_texts: List[str],
    target_langs: List[str],
):
    
    inputs = []
    for target_text, target_lang, ref_audio, ref_text in zip(
        target_texts, 
        target_langs,
        ref_audios,
        ref_texts,
    ):
        prompt = f"<|im_start|>assistant\n{target_text}<|im_end|>\n<|im_start|>assistant\n"
        print(prompt)
        inputs.append(
            {
                "prompt": prompt,
                "additional_information": {
                    "task_type": ["Base"],
                    "ref_audio": [ref_audio],
                    "ref_text": [ref_text],
                    "text": [target_text],
                    "language": [target_lang],
                    "x_vector_only_mode": [False],
                    "max_new_tokens": [8192],
                },
            }
        )
    
    return QueryResult(
        inputs=inputs,
        model_name="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    )



target_texts = [ds[0]['tgt_sent'][i] for i in range(20)]
ref_audios = ["/lustre/users/rkoshkin/s2st/assets/ru_ref.sample.wav"] * len(target_texts)
ref_texts = [
    "Привет! С вами Программный Комитет - шоу подкаст-студии Термин-Вокс и IT-конфереции Стачка. Меня зовут Сергей Пихин.В этом подкасте мы обсуждаем главные тренды в IT-индустрии и в смежных областях. Помогают нам в этом топовые эксперты, которые делятся своими знаниями и экспертизой.",
] * len(target_texts)
target_langs = ["Russian"] * len(target_texts)


query_result = get_base_query(ref_audios, ref_texts, target_texts, target_langs)

sampling_params = SamplingParams(
    temperature=0.9,
    top_p=1.0,
    top_k=50,
    max_tokens=8192,
    seed=42,
    detokenize=False,
    repetition_penalty=1.05,
)

sampling_params_list = [
    sampling_params,
]
output_dir = "/lustre/users/rkoshkin/vllm-omni/examples/offline_inference/qwen3_tts/output"
os.makedirs(output_dir, exist_ok=True)

omni_generator = omni.generate(query_result.inputs, sampling_params_list)
for stage_outputs in omni_generator:
    for output in stage_outputs.request_output:
        request_id = output.request_id
        audio_tensor = output.outputs[0].multimodal_output["audio"].clone()
        print(f"audio_tensor: {audio_tensor.shape}")
        output_wav = os.path.join(output_dir, f"output_{request_id}.wav")
        audio_samplerate = output.outputs[0].multimodal_output["sr"].item()
        # Convert to numpy array and ensure correct format
        audio_numpy = audio_tensor.float().detach().cpu().numpy()

        # Ensure audio is 1D (flatten if needed)
        if audio_numpy.ndim > 1:
            audio_numpy = audio_numpy.flatten()

        # Save audio file with explicit WAV format
        sf.write(output_wav, audio_numpy, samplerate=audio_samplerate, format="WAV")
        print(f"Request ID: {request_id}, Saved audio to {output_wav}")

<|im_start|>assistant
Добро пожаловать в новый эпизод «Out of the Pods».<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Я Дип Т. И я Натали.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
И с хорошей средыю.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Вы знаете, мы сказали на прошлой неделе, что этот эпизод будет посвящён нашему обзору «Perfect Match» Сезона 2, эпизодов 1‑6, о которых мы разберёмся.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Много мыслей.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
На самом деле почти никаких мыслей, потому что…<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Это не отличный сезон.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Это просто не началось хорошо.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
Мне кажется, я потерял несколько мозговых клеток, наблюдая это.<|im_end|>
<|im_start|>assistant

<|im_start|>assistant
О, 100 %.<|im_end|>
<|im_start|>assistant

<|im_start|>assis

Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:20 [configuration_qwen3_tts.py:441] code_predictor_config is None. Initializing code_predictor model with default values


(Worker pid=1593152) Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


(Worker pid=1593152) [Stage-0] INFO 02-21 00:54:25 [configuration_qwen3_tts.py:441] code_predictor_config is None. Initializing code_predictor model with default values


(Worker pid=1593152) Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863] WorkerProc hit an exception.
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863] Traceback (most recent call last):
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863]   File "/lustre/users/rkoshkin/vllm/vllm/v1/executor/multiproc_executor.py", line 858, in worker_busy_loop
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863]     output = func(*args, **kwargs)
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863]   File "/lustre/users/rkoshkin/vllm-omni/.venv/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863]     return func(*args, **kwargs)
(Worker pid=1593152) [Stage-0] ERROR 02-21 00:54:28 [multiproc_executor.py:863]   File "/lustre/users/rkoshkin/vllm/vllm/v1/worker/gpu_worker.py",

(EngineCore_DP0 pid=1592350) Process EngineCore_DP0:
(EngineCore_DP0 pid=1592350) Traceback (most recent call last):
(EngineCore_DP0 pid=1592350)   File "/home/user02168/.local/share/uv/python/cpython-3.10.16-linux-x86_64-gnu/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=1592350)     self.run()
(EngineCore_DP0 pid=1592350)   File "/home/user02168/.local/share/uv/python/cpython-3.10.16-linux-x86_64-gnu/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=1592350)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=1592350)   File "/lustre/users/rkoshkin/vllm/vllm/v1/engine/core.py", line 1010, in run_engine_core
(EngineCore_DP0 pid=1592350)     raise e
(EngineCore_DP0 pid=1592350)   File "/lustre/users/rkoshkin/vllm/vllm/v1/engine/core.py", line 999, in run_engine_core
(EngineCore_DP0 pid=1592350)     engine_core.run_busy_loop()
(EngineCore_DP0 pid=1592350)   File "/lustre/users/rkoshkin/vllm/vllm/v1/eng

In [20]:
!rm output/*

rm: cannot remove 'output/*': No such file or directory


In [4]:
[ds[0]['src_sent'][i] for i in range(10)]

['Welcome to another episode of Out of the Pods.',
 "I'm Deep T. And I'm Natalie.",
 'And happy Wednesday.',
 'You know, we said last week that this episode is going to be about our recap of Perfect Match Season 2, Episodes 1 through 6, which we will get into.',
 'Lots of thoughts.',
 'Actually, almost no thoughts because...',
 'This is not a great season.',
 "It's just not off to a good start.",
 'I feel like I lost some brain cells watching it.',
 'Oh, 100%.']